In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Konfigurowanie ograniczania błędów

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Dostępna jest już wersja beta nowego modelu wykonania. Ukierunkowany model wykonania zapewnia większą elastyczność podczas dostosowywania przepływu pracy ograniczania błędów. Więcej informacji znajdziesz w przewodniku [Ukierunkowany model wykonania](/guides/directed-execution-model).


<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Techniki ograniczania błędów umożliwiają użytkownikom zmniejszenie błędów Circuit przez
modelowanie szumów urządzenia w czasie wykonywania. Zazwyczaj
wiąże się to z kwantowym narzutem przetwarzania wstępnego związanym z uczeniem modelu oraz
klasycznym narzutem przetwarzania końcowego, który służy do ograniczania błędów w surowych wynikach
przy użyciu wygenerowanego modelu.

Prymityw Estimator obsługuje kilka technik ograniczania błędów, w tym [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec) oraz [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Opis każdej z nich znajdziesz w artykule [Techniki ograniczania i tłumienia błędów](error-mitigation-and-suppression-techniques). Używając prymitywów, możesz włączać lub wyłączać poszczególne metody. Szczegóły opisano w sekcji [Niestandardowe ustawienia błędów](#advanced-error).

> **Note:** Sampler nie obsługuje ograniczania błędów, ale możesz użyć pakietu [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (mitygacja pomiaru bez macierzy) do lokalnego ograniczania błędów.

Estimator obsługuje też `resilience_level`. Poziom odporności określa, jak duże zabezpieczenie przed
błędami ma być zastosowane. Wyższe poziomy dają dokładniejsze wyniki kosztem
dłuższego czasu przetwarzania. Poziomy odporności pozwalają skonfigurować
kompromis między kosztem a dokładnością przy stosowaniu ograniczania błędów w zapytaniu do prymitywu. Ograniczanie błędów redukuje błędy (odchylenie) w wynikach przez przetwarzanie
wyjść z kolekcji – czyli zbioru – powiązanych Circuit. Stopień redukcji błędów zależy od zastosowanej metody. Poziom odporności abstrakcyjnie ujmuje szczegółowy wybór metody ograniczania błędów, dzięki czemu
użytkownicy mogą oceniać kompromis kosztów i dokładności odpowiedni
dla danej aplikacji.

W związku z tym każdy poziom odpowiada metodzie lub metodom o
rosnącym kwantowym narzucie próbkowania, co pozwala eksperymentować
z różnymi kompromisami czasu i dokładności. Poniższa tabela pokazuje,
które poziomy i odpowiadające im metody są dostępne dla poszczególnych
prymitywów.

> **Info:** Ograniczanie błędów jest zadaniowo specyficzne, dlatego dostępne techniki
> różnią się w zależności od tego, czy próbkujesz rozkład, czy generujesz
> wartości oczekiwane.

<span id="resilience-table"></span>

Estimator obsługuje następujące poziomy odporności. Sampler nie obsługuje poziomów odporności.

| Poziom odporności | Definicja                                                                                                      | Technika                                                                  |
|-------------------|----------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                 | Brak mitygacji                                                                                                 | Brak                                                                      |
| 1 [Domyślny]      | Minimalne koszty mitygacji: ograniczanie błędów związanych z błędami odczytu                                  | Twirled Readout Error eXtinction (TREX) – splatanie pomiarów              |
| 2                 | Średnie koszty mitygacji. Zazwyczaj redukuje odchylenie w estymatorach, ale nie gwarantuje wyniku bez odchylenia. | Poziom 1 + Zero Noise Extrapolation (ZNE) i splatanie bramek

> **Info:** Poziomy odporności są obecnie w wersji beta, dlatego narzut próbkowania i
> jakość rozwiązania mogą się różnić w zależności od Circuit. Nowe funkcje,
> zaawansowane opcje i narzędzia zarządzania będą udostępniane sukcesywnie. Nie ma gwarancji,
> że na każdym poziomie odporności zostaną zastosowane konkretne metody ograniczania błędów.

## Konfigurowanie Estimatora z poziomami odporności
Możesz używać poziomów odporności do określania technik ograniczania błędów albo ustawiać niestandardowe techniki indywidualnie, jak opisano w sekcji [Niestandardowe ustawienia błędów.](#advanced-error)

<details>
<summary>Poziom odporności 0</summary>

Do programu użytkownika nie jest stosowane żadne ograniczanie błędów.

</details>

<details>
<summary>Poziom odporności 1</summary>

Poziom 1 stosuje **mitygację błędów odczytu** i **splatanie pomiarów** za pomocą wolnej od modelu techniki
zwanej Twirled Readout Error eXtinction (TREX). Redukuje błąd pomiaru
przez diagonalizację kanału szumów związanego z pomiarem poprzez
losowe odwracanie Qubitów przez bramki X tuż przed pomiarem. Termin
przeskalowania z diagonalnego kanału szumów jest wyznaczany przez
benchmarking losowych Circuit zainicjalizowanych w stanie zerowym. Dzięki temu
usługa może usuwać odchylenie z wartości oczekiwanych wynikających z
szumu odczytu. Podejście to jest szerzej opisane w artykule [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Poziom odporności 2</summary>

Poziom 2 stosuje **techniki ograniczania błędów zawarte w poziomie 1**, a ponadto stosuje **splatanie bramek** i używa **metody Zero Noise Extrapolation (ZNE)**. ZNE oblicza
wartość oczekiwaną obserwowalnej dla różnych czynników szumu
(etap amplifikacji), a następnie używa zmierzonych wartości oczekiwanych do
wyznaczenia idealnej wartości oczekiwanej przy zerowym szumie (etap
ekstrapolacji). Podejście to zazwyczaj redukuje błędy w wartościach oczekiwanych, ale
nie gwarantuje wyniku bez odchylenia.

![Ten obraz przedstawia wykres. Oś X jest opisana jako Współczynnik amplifikacji szumu. Oś Y jest opisana jako Wartość oczekiwana. Linia wznosząca się jest oznaczona jako Wartość mitygowana. Punkty blisko linii to wartości wzmocnione szumem. Nad osią X przebiega pozioma linia oznaczona jako Wartość dokładna.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Ilustracja metody ZNE")

Narzut tej metody skaluje się wraz z liczbą czynników szumu. Domyślne
ustawienia próbkują wartość oczekiwaną przy trzech czynnikach szumu,
co prowadzi do około 3-krotnego narzutu przy stosowaniu tego poziomu odporności.

Na poziomie 2 metoda TREX losowo odwraca Qubity przez bramki X tuż przed pomiarem
i odwraca odpowiedni zmierzony bit, jeśli zastosowano bramkę X. Podejście to jest szerzej opisane w artykule [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Przykład
Interfejs `EstimatorV2` pozwala użytkownikom bezproblemowo korzystać z różnych
metod ograniczania błędów w celu redukcji błędów w wartościach oczekiwanych
obserwowalnych. Poniższy kod używa metody Zero Noise Extrapolation i mitygacji błędów odczytu przez proste
ustawienie `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Niestandardowe ustawienia błędów
Możesz włączać i wyłączać poszczególne metody ograniczania i tłumienia błędów, w tym dynamiczne rozsprzęganie, splatanie bramek i pomiarów, mitygację błędów pomiarów, PEC i ZNE. Opis każdej z nich znajdziesz w artykule [Techniki ograniczania i tłumienia błędów](error-mitigation-and-suppression-techniques).

> **Note:** - Nie wszystkie opcje są dostępne dla obu prymitywów. Listę dostępnych opcji znajdziesz w [tabeli dostępnych opcji](runtime-options-overview#options-table).
> - Nie wszystkie metody działają razem na wszystkich typach Circuit. Szczegóły znajdziesz w [tabeli zgodności funkcji](runtime-options-overview#options-compatibility-table).

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">